# Demo 2 · Data exploration and visualization

This notebook covers **data exploration** and **visualization and exploratory data science**.
Inspect raw NHL events with a slider, replay shots on a rink, and use an interactive
cumulative-shot chart to discuss how the game develops.

Start with [From an API to a pandas table](01_data_acquisition_and_cleaning.ipynb)
for the acquisition and cleaning lesson. This follow-up runs in a fresh kernel:
the short setup below loads the same game and rebuilds the same shot table.


## Setup: choose local or Colab

**Locally:** run `uv sync` from the `ift3700-6758` repository root, then select the shared Python 3.11 `.venv` kernel. See the [repository README](../../../README.md) for setup. Skip the optional installation cell: it installs nothing locally.

**Google Colab | no repository clone or manual uv installation needed:**

1. Open this notebook in Colab and choose a **CPU** runtime. Installation needs internet access.
2. In the **optional cell below**, set `INSTALL_COLAB_PACKAGES = True` and run it once. The cell installs `uv`, then uses it to install the libraries into the notebook's current Python. `sys.executable` is that Python's path; `subprocess.check_call` runs a command and stops if it fails.
3. If Colab requests a restart after installation, choose **Runtime → Restart session**.
4. Set the flag back to `False`, then run the cells from top to bottom with **Shift+Enter**. Repeat installation when Colab gives you a new runtime; a normal session restart keeps installed packages.

This notebook uses **requests** for HTTP and **pandas** for tables; `json` and `pathlib` come with Python. **Plotly** creates interactive charts and **ipywidgets** supplies the sliders. The cell also enables Colab widgets, even when installation is disabled.

The first run needs internet access to download the game. Later runs reuse the saved JSON file.


In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "pandas>=2.2,<4",
        "requests>=2.31,<3",
        "plotly>=6,<7",
        "ipywidgets>=8.1,<9",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

if IN_COLAB:
    # Enable interactive sliders, including after a session restart.
    output.enable_custom_widget_manager()


### The data folder
`Path` constructs paths. In Colab we use the current folder.
Locally we locate the repository root to reuse its cache.
JSON lives in `data/raw`; exports go into `data/processed`.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not IN_COLAB:
    # Walk up to the local repository.
    for folder in [ROOT, *ROOT.parents]:
        if (folder / "demo_2").is_dir():
            ROOT = folder
            break
os.chdir(ROOT)
print(ROOT / "data/raw/2025030311.json")

## Prepare the data for exploration

This is a compact recap of part 1, using the same game and cleaning rules.
Load the cached raw JSON, or download it once if it is absent; flatten the events,
keep shots and goals, and check their identifiers. No variables from another
notebook are required. The original acquisition notebook is unchanged.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

game_id = 2025030311
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
print(url)

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
raw_path = raw_dir / f"{game_id}.json"
print("Raw data file:", raw_path.resolve())

if raw_path.exists():
    print("Already downloaded — reusing", raw_path.name)
else:
    response = requests.get(url, timeout=30)
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    response.raise_for_status()
    downloaded_game = response.json()

    # Check we received the requested game's event data before saving it.
    assert downloaded_game["id"] == game_id
    assert isinstance(downloaded_game["plays"], list)
    raw_path.write_text(json.dumps(downloaded_game, indent=2), encoding="utf-8")
    print("Saved", raw_path.name)

game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == game_id
print("Python type:", type(game))
print("Top-level keys:", list(game.keys()))

In [ ]:
plays = game["plays"]
events = pd.json_normalize(plays)
print("Rows, columns:", events.shape)
print("Columns:", events.columns.tolist())
display(events.head())

keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])
shots = events.loc[keep].copy()
print("All events:", len(events))
print("Retained shots and goals:", len(shots))
print("Other events left out:", len(events) - len(shots))

column_names = {
    "eventId": "event_id",
    "periodDescriptor.number": "period",
    "periodDescriptor.periodType": "period_type",
    "timeInPeriod": "time_in_period",
    "typeDescKey": "event_type",
    "details.eventOwnerTeamId": "team_id",
    "details.xCoord": "x",
    "details.yCoord": "y",
    "details.shotType": "shot_type",
}
shots = shots.reindex(columns=list(column_names)).rename(columns=column_names)
shots["game_id"] = game["id"]
shots["season"] = game["season"]
shots["game_type"] = game["gameType"]
shots["is_goal"] = shots["event_type"].eq("goal")

team_names = {
    game["awayTeam"]["id"]: game["awayTeam"]["abbrev"],
    game["homeTeam"]["id"]: game["homeTeam"]["abbrev"],
}
shots["team"] = shots["team_id"].map(team_names)
shots = shots.convert_dtypes()
display(shots.head(10))

assert len(events) == len(plays), "Flattening changed the number of events."
assert len(shots) == int(keep.sum()), "Cleaning changed the number of retained events."
assert shots[["game_id", "event_id"]].notna().all().all(), "Missing event identifier."
assert not shots.duplicated(["game_id", "event_id"]).any(), "Duplicate event identifiers."

print("Checks passed.")
display(shots.isna().sum().rename("missing_count").to_frame())

## Objective 2 · inspect before plotting
`events` contains all events; `shots` contains shots on goal **including goals**,
prepared with part 1's cleaning rules. Start by counting event types.
What does one row represent?


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from ipywidgets import interact, IntSlider

LANG = 'en'
pio.renderers.default = "colab" if IN_COLAB else "plotly_mimetype+notebook"
teams = [game["awayTeam"]["abbrev"], game["homeTeam"]["abbrev"]]
colors = dict(zip(teams, ["#2364ce", "#e25c45"]))

display(events["typeDescKey"].value_counts().rename("count").to_frame())
if LANG == "fr":
    print(f"{len(events)} événements → {len(shots)} tirs cadrés, buts inclus")
else:
    print(f"{len(events)} events → {len(shots)} shots on goal, goals included")

### A small reusable function: draw the rink
A function groups instructions. This one adds a rectangle and three lines to a
Plotly figure, then returns it. Fixed axes preserve the proportions.
Coordinates are in feet. This is a schematic, not a hockey-specific package.

In [ ]:
def draw_rink(fig):
    # A 200-foot by 85-foot rectangle.
    fig.add_shape(type="rect", x0=-100, x1=100, y0=-42.5, y1=42.5,
                  line_color="#7f95aa")
    for x, colour in [(-25, "#9bc2e6"), (0, "#e6a0a0"), (25, "#9bc2e6")]:
        fig.add_vline(x=x, line_color=colour)
    fig.update_xaxes(range=[-103, 103], title="x (ft)", constrain="domain")
    fig.update_yaxes(range=[-46, 46], title="y (ft)", scaleanchor="x", scaleratio=1)
    fig.update_layout(template="plotly_white", height=460,
                      margin=dict(l=55, r=25, t=55, b=50))
    return fig

### A slider becomes a function argument
`@interact` calls the function whenever the slider changes.
`i` is a **list position**, not the NHL event identifier.
The table below lists goal and period-start positions to try.
Without coordinates, show the JSON and an empty rink, never a fake point at `(0, 0)`.

In [ ]:
examples = events["typeDescKey"].isin(["goal", "period-start"])
display(events.loc[examples, ["eventId", "typeDescKey"]])

@interact(i=IntSlider(min=0, max=len(game["plays"])-1, continuous_update=False))
def inspect_event(i):
    play = game["plays"][i]
    print(json.dumps(play, indent=2, ensure_ascii=False))
    details = play.get("details", {})
    x = details.get("xCoord")
    y = details.get("yCoord")
    fig = go.Figure()
    # Plot only if both coordinates exist.
    if x is not None and y is not None:
        fig.add_scatter(x=[x], y=[y], mode="markers", marker_size=18)
    fig.update_layout(title=f"eventId={play['eventId']} · {play['typeDescKey']}")
    draw_rink(fig).show()

## Objective 3 · build a clock
`mm:ss` is elapsed time within the period. For periods 1–3:
`minute = (period - 1) × 20 + mm + ss / 60`.
A copy named `timed` excludes overtime and shootouts; `shots` stays intact.

In [ ]:
regular = (shots["period_type"] == "REG") & shots["period"].between(1, 3)
timed = shots.loc[regular].copy()
clock = timed["time_in_period"].str.split(":", expand=True).astype(float)
timed["minute"] = (timed["period"] - 1) * 20 + clock[0] + clock[1] / 60
timed = timed.sort_values(["minute", "event_id"])
label = "Tirs exclus du replay :" if LANG == "fr" else "Shots excluded from replay:"
print(label, len(shots) - len(timed))
display(timed[["period", "time_in_period", "minute", "team"]].head())

### The game in one minute
The ▶ button advances the slider. At each minute, filter shots that have already happened.
`color` identifies the team; `symbol` distinguishes goals (stars).
Pause at minute 20 and predict what comes next. This is not puck tracking.

In [ ]:
import ipywidgets as widgets

# Link the play button to the slider.
play = widgets.Play(min=0, max=60, interval=500)
minute_slider = IntSlider(min=0, max=60, continuous_update=False)
widgets.jslink((play, "value"), (minute_slider, "value"))
display(play)

@interact(minute=minute_slider)
def replay_at(minute):
    visible = timed.loc[timed["minute"] <= minute].dropna(subset=["x", "y"])
    title = f"Minute {minute} · {len(visible)} tirs" if LANG == "fr" else f"Minute {minute} · {len(visible)} shots"
    fig = px.scatter(visible, x="x", y="y", color="team", symbol="is_goal",
                     color_discrete_map=colors, symbol_map={False: "circle", True: "star"},
                     category_orders={"team": teams},
                     hover_data=["period", "time_in_period"], title=title)
    fig.update_traces(marker_size=13, marker_opacity=0.85)
    draw_rink(fig).show()

### From animation to analysis: cumulative shots
`cumcount() + 1` numbers each team's shots in chronological order.
A step curve does not suggest shots between events.
A steep slope means many shots, not high possession.

In [ ]:
timed["cumulative_shots"] = timed.groupby("team").cumcount() + 1
fig = go.Figure()
for team, group in timed.groupby("team"):
    # Include the start and end of the game.
    minutes = [0] + group["minute"].tolist() + [60]
    totals = [0] + group["cumulative_shots"].tolist() + [len(group)]
    fig.add_scatter(x=minutes, y=totals, name=team, mode="lines",
                    line=dict(shape="hv", color=colors[team]))
    goals = group.loc[group["is_goal"]]
    fig.add_scatter(x=goals["minute"], y=goals["cumulative_shots"],
                    mode="markers", showlegend=False,
                    marker=dict(symbol="star", size=14, color=colors[team]),
                    hovertemplate=team + " · goal<extra></extra>")
for minute in [20, 40]:
    fig.add_vline(x=minute, line_dash="dot")
y_title = "Tirs cadrés cumulés" if LANG == "fr" else "Cumulative shots on goal"
title = "Quand les tirs s'accumulent-ils ?" if LANG == "fr" else "When do shots accumulate?"
fig.update_layout(title=title, xaxis_title="Minutes", yaxis_title=y_title,
                  template="plotly_white", hovermode="x unified")
fig.show()

### Your turn · 3 minutes
Count each team's shots between minutes 15 and 20.

In [ ]:
window = timed.loc[timed["minute"].between(15, 20)]
# Complete here.

<details><summary>Reference answer</summary>

```python
display(window.groupby("team").size())
```

</details>

## Export and takeaway
`fig` is the last main figure. Its HTML includes Plotly and opens without a Python kernel.
Widgets need an active kernel. In Colab, download the export through the Files panel.
One game illustrates the method; the milestone requires multiple games/seasons and hourly excess maps.
**Lecture 6:** slides 5 (explore/present), 9 (question first), 20–21 (one question per figure), 47 (interactivity and uncertainty).

In [ ]:
output_dir = ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
target = output_dir / "figure_b_en.html"
fig.write_html(target, include_plotlyjs=True)
print(target)